# Étape 1 — OCR sur un dossier de test (Module 2)

Objectif de cette étape :
- Rasteriser un PDF de dossier (Dataset_maladie ou Dataset_dentaire) en images, page par page
- Faire tourner PaddleOCR sur chaque page
- Sauvegarder un JSON structuré : texte + score de confiance par ligne, par page
- Faire des vérifications OBJECTIVES (pas juste "je regarde si ça a l'air bien") :
    * nombre de pages traitées == nombre de pages du PDF
    * confiance moyenne par page (détecter les pages mal scannées)
    * comparaison avec ground_truth.json si un nom de dossier attendu existe

Cette étape est volontairement isolée : elle ne fait QUE de l'OCR brut.
L'extraction des champs (nom, CIN, numéro d'adhérent...) sera l'étape 2.

In [1]:
import pymupdf as fitz  # PyMuPDF — alias fitz conservé pour la lisibilité du reste du code
from paddleocr import PaddleOCR
import numpy as np
import cv2
import json
import time
from pathlib import Path

c:\Users\DELL\Desktop\PROJET_PFA\PFA-Module2-ControleDocumentaire\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration — à adapter à ton environnement

In [2]:
# Chemin vers UN dossier de test (change selon ce que tu veux tester)
DATASET_DIR = Path("../data/Dataset_maladie")
DOSSIER_TEST = DATASET_DIR / "Dossier_000001.pdf"
GROUND_TRUTH_PATH = DATASET_DIR / "ground_truth.json"

OUTPUT_DIR = Path("../outputs/ocr_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Résolution de rasterisation : 200-300 DPI est un bon compromis qualité/vitesse pour de l'OCR
DPI = 250

assert DOSSIER_TEST.exists(), f"Fichier introuvable : {DOSSIER_TEST}"

## Initialisation de PaddleOCR

`lang='fr'` gère le français ; `use_angle_cls=True` corrige les pages
scannées de travers (fréquent dans un flux de numérisation réel).

In [3]:
# Initialisation compatible avec l'API PaddleOCR 3.x, optimisée pour CPU :
# - use_angle_cls et show_log ont été retirés/renommés dans les versions récentes
# - use_textline_orientation remplace use_angle_cls (correction des lignes tournées)
# - use_doc_orientation_classify / use_doc_unwarping désactivés (scans déjà droits)
# - enable_mkldnn=False : contourne un bug connu de PaddlePaddle 3.3.0 sur CPU
#   (NotImplementedError: ConvertPirAttribute2RuntimeAttribute...)
# - modèles 'mobile' au lieu du défaut 'PP-OCRv6_medium' : bien plus légers et
#   rapides sur CPU, seule option viable sans GPU pour un usage en développement
# - cpu_threads : ajuste selon le nombre de coeurs de ta machine
import os
ocr_engine = PaddleOCR(
    lang='fr',
    use_textline_orientation=True,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    enable_mkldnn=False,
    text_detection_model_name='PP-OCRv5_mobile_det',
    text_recognition_model_name='PP-OCRv5_mobile_rec',
    cpu_threads=os.cpu_count(),
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_16084\4189704422.py:11: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  ocr_engine = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_mobile_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-OCRv5_mobile_rec`.


## Fonction : PDF → liste d'images (une par page)

In [4]:
def pdf_vers_images(pdf_path: Path, dpi: int = 250) -> list[np.ndarray]:
    """Rasterise chaque page d'un PDF en image numpy (BGR, pour OpenCV/PaddleOCR)."""
    doc = fitz.open(pdf_path)
    zoom = dpi / 72  # PyMuPDF travaille en points (72 dpi de base)
    mat = fitz.Matrix(zoom, zoom)
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        if pix.n == 4:  # RGBA -> BGR
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
        elif pix.n == 3:  # RGB -> BGR
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        images.append(img)
    doc.close()
    return images

## Fonction : OCR d'une page + extraction structurée

In [5]:
def _resultat_dict(res):
    """
    Normalise l'accès au résultat PaddleOCR 3.x, qui se comporte comme un
    dict mais expose parfois ses champs sous une clé 'res' selon la version.
    """
    if isinstance(res, dict) and "rec_texts" not in res and "res" in res:
        return res["res"]
    return res


def ocr_page(image: np.ndarray) -> dict:
    """Retourne le texte et les scores de confiance d'une page (API PaddleOCR 3.x)."""
    resultats = ocr_engine.predict(image)
    res = _resultat_dict(resultats[0]) if resultats else {}

    textes = res.get("rec_texts", [])
    scores = res.get("rec_scores", [])
    polys = res.get("rec_polys", res.get("dt_polys", [None] * len(textes)))

    lignes = []
    for texte, score, poly in zip(textes, scores, polys):
        bbox = poly.tolist() if hasattr(poly, "tolist") else poly
        lignes.append({
            "texte": texte,
            "confiance": round(float(score), 4),
            "bbox": bbox,
        })

    confiances = [l["confiance"] for l in lignes]
    return {
        "lignes": lignes,
        "nb_lignes": len(lignes),
        "confiance_moyenne": round(float(np.mean(confiances)), 4) if confiances else 0.0,
        "confiance_min": round(float(np.min(confiances)), 4) if confiances else 0.0,
    }

## Exécution sur le dossier de test

In [6]:
t0 = time.time()

images = pdf_vers_images(DOSSIER_TEST, dpi=DPI)
print(f"Nombre de pages rasterisées : {len(images)}")

resultats_dossier = {
    "dossier": DOSSIER_TEST.name,
    "nb_pages": len(images),
    "pages": []
}

for i, img in enumerate(images):
    print(f"  OCR page {i + 1}/{len(images)}...")
    resultat_page = ocr_page(img)
    resultat_page["page"] = i + 1
    resultats_dossier["pages"].append(resultat_page)

duree = time.time() - t0
resultats_dossier["duree_secondes"] = round(duree, 2)

print(f"\nTerminé en {duree:.1f}s")

Nombre de pages rasterisées : 4
  OCR page 1/4...
  OCR page 2/4...
  OCR page 3/4...
  OCR page 4/4...

Terminé en 191.9s


## Sauvegarde du résultat brut

In [7]:
output_path = OUTPUT_DIR / f"{DOSSIER_TEST.stem}_ocr.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(resultats_dossier, f, ensure_ascii=False, indent=2)

print(f"Résultat sauvegardé : {output_path}")

Résultat sauvegardé : ..\outputs\ocr_results\Dossier_000001_ocr.json


## Vérifications OBJECTIVES (pas visuelles)

Ces contrôles doivent être lus et compris avant de passer à l'étape 2.
Un score de confiance moyen anormalement bas indique un problème de
rasterisation (DPI trop faible, page tournée) plutôt qu'un vrai problème
de contenu — à corriger ICI avant d'aller plus loin.

In [8]:
print("=== Vérifications objectives ===\n")

# 1. Cohérence du nombre de pages
doc_check = fitz.open(DOSSIER_TEST)
nb_pages_pdf = doc_check.page_count
doc_check.close()
assert nb_pages_pdf == len(images), (
    f"Incohérence : {nb_pages_pdf} pages dans le PDF vs {len(images)} images générées"
)
print(f"[OK] Nombre de pages cohérent : {nb_pages_pdf}")

# 2. Confiance moyenne par page — seuil d'alerte à ajuster après quelques essais
SEUIL_ALERTE_CONFIANCE = 0.70
pages_a_verifier = []
for p in resultats_dossier["pages"]:
    statut = "OK" if p["confiance_moyenne"] >= SEUIL_ALERTE_CONFIANCE else "A VERIFIER"
    if statut == "A VERIFIER":
        pages_a_verifier.append(p["page"])
    print(f"  Page {p['page']}: {p['nb_lignes']} lignes, "
          f"confiance moyenne = {p['confiance_moyenne']:.3f} [{statut}]")

if pages_a_verifier:
    print(f"\n[ATTENTION] Pages à inspecter manuellement : {pages_a_verifier}")
else:
    print("\n[OK] Toutes les pages dépassent le seuil de confiance.")

# 3. Comparaison avec ground_truth.json (si disponible) — vérifie au minimum
#    que le nombre de pages attendu correspond à ce qu'on a détecté.
if GROUND_TRUTH_PATH.exists():
    with open(GROUND_TRUTH_PATH, "r", encoding="utf-8") as f:
        ground_truth = json.load(f)

    # ground_truth.json est une LISTE de dicts (un par dossier), chacun avec
    # un champ "fichier" -- pas un dict indexé par nom de fichier.
    entree_gt = next(
        (d for d in ground_truth if d.get("fichier") == DOSSIER_TEST.name),
        None
    )
    if entree_gt is not None:
        print(f"\n[INFO] Entrée ground truth trouvée pour {DOSSIER_TEST.name}")
        print(json.dumps(entree_gt, ensure_ascii=False, indent=2)[:500])
    else:
        print(f"\n[INFO] Pas d'entrée ground truth pour la clé '{DOSSIER_TEST.name}' "
              f"— vérifie le format des clés dans ground_truth.json")
else:
    print(f"\n[INFO] ground_truth.json non trouvé à {GROUND_TRUTH_PATH} — vérification ignorée")

=== Vérifications objectives ===

[OK] Nombre de pages cohérent : 4
  Page 1: 113 lignes, confiance moyenne = 0.896 [OK]
  Page 2: 25 lignes, confiance moyenne = 0.847 [OK]
  Page 3: 18 lignes, confiance moyenne = 0.714 [OK]
  Page 4: 101 lignes, confiance moyenne = 0.708 [OK]

[OK] Toutes les pages dépassent le seuil de confiance.

[INFO] Entrée ground truth trouvée pour Dossier_000001.pdf
{
  "fichier": "Dossier_000001.pdf",
  "nombre_pages": 4,
  "categories": [
    "feuille_principale",
    "facture",
    "ordonnance",
    "fiche_pharmacique"
  ],
  "coherent": true,
  "type_erreur": null
}


## Prochaine étape

Une fois que tu as validé ceci sur 3-5 dossiers différents (varier
maladie/dentaire, dossiers "propres" et dossiers marqués incohérents dans
le ground truth), passe à `02_field_extraction.py` : extraction du nom,
prénom, CIN, numéro d'adhérent, numéro de dossier, date de naissance à
partir du texte brut OCR obtenu ici (regex + position sur la page).

## Traitement par lot (plusieurs dossiers d'un même dataset)

Pour l'évaluation précision/rappel, on a besoin de traiter 20 à 50 dossiers
d'un coup — mais **jamais les deux datasets mélangés dans le même lot**.

Deux garde-fous pour ça :
- `NOM_DATASET` choisit explicitement `"maladie"` OU `"dentaire"`, jamais
  les deux à la fois dans une même exécution de cette section.
- La liste exacte des dossiers à traiter vient du fichier
  `outputs/echantillon_test_<dataset>.txt`, généré par
  `00_exploration_dataset.ipynb` (tirage aléatoire, dossiers déjà
  utilisés pour ajuster le code exclus). Ce n'est plus "les N
  premiers fichiers triés par nom".
- Les résultats sont sauvegardés dans un sous-dossier séparé par dataset
  (`outputs/ocr_results/maladie/` ou `.../dentaire/`), et PAS directement
  dans `outputs/ocr_results/`. C'est indispensable : les deux datasets
  contiennent des fichiers de même nom (`Dossier_000001.pdf` existe dans
  les deux), donc sans ce sous-dossier, traiter l'un après l'autre
  écraserait silencieusement les résultats du premier.

⚠️ **Estimation de durée** : environ 3 à 4 minutes par dossier de 4 pages
sur CPU (voir le test à dossier unique plus haut). Un lot de 20 dossiers
prend donc environ **1h à 1h20**, et un lot de 50 dossiers environ
**2h30 à 3h20**. Lance ça quand tu peux laisser tourner la machine sans
l'interrompre (le kernel doit rester actif jusqu'à la fin).

In [12]:
NOM_DATASET = "maladie"  # "maladie" ou "dentaire" -- jamais les deux à la fois

DATASET_DIR_LOT = Path(f"../data/Dataset_{NOM_DATASET}")
OUTPUT_DIR_LOT = Path(f"../outputs/ocr_results/{NOM_DATASET}")
OUTPUT_DIR_LOT.mkdir(parents=True, exist_ok=True)

assert DATASET_DIR_LOT.exists(), f"Dossier introuvable : {DATASET_DIR_LOT}"

# La liste des dossiers à traiter vient du notebook 00 (échantillon aléatoire,
# excluant les dossiers déjà utilisés pour ajuster le code) -- PAS des N
# premiers fichiers triés par nom, qui incluraient des dossiers déjà vus.
CHEMIN_ECHANTILLON = Path(f"../outputs/echantillon_test_{NOM_DATASET}.txt")
assert CHEMIN_ECHANTILLON.exists(), (
    f"{CHEMIN_ECHANTILLON} introuvable -- lance d'abord 00_exploration_dataset.ipynb "
    "pour générer un échantillon de test non biaisé."
)

noms_fichiers = CHEMIN_ECHANTILLON.read_text(encoding="utf-8").splitlines()
fichiers_pdf = [DATASET_DIR_LOT / nom for nom in noms_fichiers]

manquants = [f for f in fichiers_pdf if not f.exists()]
if manquants:
    print(f"ATTENTION : {len(manquants)} fichier(s) de l'échantillon introuvables dans {DATASET_DIR_LOT} :")
    for f in manquants:
        print(f"  {f.name}")
fichiers_pdf = [f for f in fichiers_pdf if f.exists()]

print(f"{len(fichiers_pdf)} dossiers sélectionnés (échantillon de {CHEMIN_ECHANTILLON.name})")
print(f"Résultats sauvegardés dans : {OUTPUT_DIR_LOT}")

49 dossiers sélectionnés (échantillon de echantillon_test_maladie.txt)
Résultats sauvegardés dans : ..\outputs\ocr_results\maladie


### Lancer le traitement du lot

Une ligne de progression s'affiche pour chaque dossier (pas le détail
ligne par ligne comme pour le test unique plus haut — ce serait illisible
sur 20-50 dossiers). Si le notebook doit être interrompu en cours de
route, les dossiers déjà traités restent sauvegardés individuellement :
relance simplement cette cellule, elle resautera ceux déjà faits.

In [14]:
import time

resume_lot = []
debut_lot = time.time()

for i, chemin_pdf in enumerate(fichiers_pdf, start=1):
    sortie = OUTPUT_DIR_LOT / f"{chemin_pdf.stem}_ocr.json"

    if sortie.exists():
        print(f"[{i}/{len(fichiers_pdf)}] {chemin_pdf.name} : déjà traité, ignoré")
        continue

    t0 = time.time()
    images = pdf_vers_images(chemin_pdf, dpi=DPI)

    resultats_dossier = {"dossier": chemin_pdf.name, "nb_pages": len(images), "pages": []}
    for p, img in enumerate(images):
        resultat_page = ocr_page(img)
        resultat_page["page"] = p + 1
        resultats_dossier["pages"].append(resultat_page)

    duree = time.time() - t0
    resultats_dossier["duree_secondes"] = round(duree, 2)

    with open(sortie, "w", encoding="utf-8") as f:
        json.dump(resultats_dossier, f, ensure_ascii=False, indent=2)

    confiances = [pg["confiance_moyenne"] for pg in resultats_dossier["pages"]]
    conf_moy = sum(confiances) / len(confiances) if confiances else 0.0
    resume_lot.append({
        "dossier": chemin_pdf.name, "nb_pages": len(images),
        "confiance_moyenne": round(conf_moy, 3), "duree": duree,
    })

    print(f"[{i}/{len(fichiers_pdf)}] {chemin_pdf.name} : {len(images)} pages, "
          f"confiance moy={conf_moy:.2f}, {duree:.0f}s")

duree_totale = time.time() - debut_lot
print(f"\nTerminé : lot traité en {duree_totale/60:.1f} min "
      f"({duree_totale/max(len(resume_lot),1):.0f}s/dossier en moyenne sur les nouveaux)")

[1/49] Dossier_000004.pdf : déjà traité, ignoré
[2/49] Dossier_000031.pdf : déjà traité, ignoré
[3/49] Dossier_000046.pdf : déjà traité, ignoré
[4/49] Dossier_000086.pdf : déjà traité, ignoré
[5/49] Dossier_000089.pdf : déjà traité, ignoré
[6/49] Dossier_000104.pdf : déjà traité, ignoré
[7/49] Dossier_000128.pdf : déjà traité, ignoré
[8/49] Dossier_000147.pdf : déjà traité, ignoré
[9/49] Dossier_000215.pdf : déjà traité, ignoré
[10/49] Dossier_000245.pdf : déjà traité, ignoré
[11/49] Dossier_000264.pdf : déjà traité, ignoré
[12/49] Dossier_000278.pdf : déjà traité, ignoré
[13/49] Dossier_000287.pdf : déjà traité, ignoré
[14/49] Dossier_000289.pdf : déjà traité, ignoré
[15/49] Dossier_000294.pdf : déjà traité, ignoré
[16/49] Dossier_000306.pdf : déjà traité, ignoré
[17/49] Dossier_000352.pdf : déjà traité, ignoré
[18/49] Dossier_000364.pdf : déjà traité, ignoré
[19/49] Dossier_000374.pdf : déjà traité, ignoré
[20/49] Dossier_000393.pdf : déjà traité, ignoré
[21/49] Dossier_000409.pdf : 

### Vérification rapide du lot

Avant de passer à l'étape d'évaluation, un coup d'œil sur les dossiers à
confiance anormalement basse — ça peut indiquer un scan de mauvaise
qualité dans le dataset, pas forcément un problème de code.

In [ ]:
SEUIL_ALERTE_LOT = 0.70

fichiers_json_lot = sorted(OUTPUT_DIR_LOT.glob("*_ocr.json"))
print(f"{len(fichiers_json_lot)} fichiers JSON présents dans {OUTPUT_DIR_LOT}\n")

dossiers_a_verifier = []
for chemin_json in fichiers_json_lot:
    with open(chemin_json, encoding="utf-8") as f:
        d = json.load(f)
    confiances = [p["confiance_moyenne"] for p in d["pages"]]
    conf_moy = sum(confiances) / len(confiances) if confiances else 0.0
    if conf_moy < SEUIL_ALERTE_LOT:
        dossiers_a_verifier.append((d["dossier"], round(conf_moy, 3)))

if dossiers_a_verifier:
    print("Dossiers à confiance basse (à inspecter visuellement) :")
    for nom, conf in dossiers_a_verifier:
        print(f"  {nom} : confiance moyenne = {conf}")
else:
    print("Tous les dossiers du lot dépassent le seuil de confiance.")

49 fichiers JSON présents dans ..\outputs\ocr_results\maladie

Tous les dossiers du lot dépassent le seuil de confiance.


: 